# Mixed binary, integer, and real optimisation with QQA

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yuma-Ichikawa/QQA4CO/blob/main/examples/09_mixed_integer_real_optimization.ipynb)

This notebook solves three increasingly realistic models: a continuous convex problem, a bounded integer problem, and a constrained factory-planning MINLP containing binary, integer, and real variables.

In [ ]:
# Install the repository version when running in Colab.
try:
    import qqa

    assert hasattr(qqa, "MixedProblem")
except (ImportError, AssertionError):
    %pip install -q "qqa @ git+https://github.com/Yuma-Ichikawa/QQA4CO.git"

In [ ]:
import itertools
from pathlib import Path

import torch

import qqa
from qqa import visualization as viz

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
qqa.fix_seed(0)
print("device:", DEVICE)

## 1. Bounded real optimisation

The objective receives named tensors and preserves the leading replica dimension.

In [ ]:
real_problem = qqa.MixedProblem(
    [qqa.Real("x", -5.0, 5.0), qqa.Real("y", -5.0, 5.0)],
    lambda v: (v["x"] - 1.25).square() + (v["y"] + 2.5).square(),
    name="convex-real",
)
real_result = real_problem.solve(sol_size=64, num_epochs=400, device=DEVICE, verbose=False)
print(real_result.best_obj, real_result.score["extra"]["variables"])

## 2. Bounded integer optimisation

The continuous search is projected onto the declared integer grid when candidates are scored.

In [ ]:
integer_problem = qqa.MixedProblem(
    [qqa.Integer("quantity", lower=-10, upper=10)],
    lambda v: (v["quantity"] - 3).square(),
    name="integer-quadratic",
)
integer_result = integer_problem.solve(sol_size=32, num_epochs=250, device=DEVICE, verbose=False)
print(integer_result.best_obj, integer_result.score["extra"]["variables"])

## 3. Practical mixed factory-planning model

Two machines have fixed activation costs. Production occurs in integer batches, while bounded real-valued overtime can close a small demand gap. Constraints enforce demand and link batches to active machines.

In [ ]:
factory = qqa.MixedProblem(
    variables=[
        qqa.Binary("machine", size=2),
        qqa.Integer("batches", lower=0, upper=6, size=2),
        qqa.Real("overtime", lower=0.0, upper=4.0),
    ],
    objective=lambda v: (
        10 * v["machine"].sum(-1) + 3 * v["batches"].sum(-1) + 2 * v["overtime"].square()
    ),
    constraints=[
        qqa.Constraint(
            lambda v: 4 * v["batches"].sum(-1) + v["overtime"],
            sense=">=",
            rhs=28,
            weight=100,
            name="demand",
        ),
        qqa.Constraint(
            lambda v: (v["batches"] - 6 * v["machine"]).clamp_min(0).sum(-1),
            sense="<=",
            rhs=0,
            weight=100,
            name="activation_link",
        ),
    ],
    name="factory-planning",
    objective_label="cost",
    objective_unit="kUSD",
)

factory_result = factory.solve(
    sol_size=256,
    num_epochs=1000,
    div_param=0.1,
    device=DEVICE,
    mixed_precision="bf16" if DEVICE == "cuda" else "fp32",
    verbose=False,
)
factory_result.score

## 4. Independent verification

Enumerating the tiny discrete part and analytically selecting the minimum required overtime confirms the global feasible optimum.

In [ ]:
best = None
for machine in itertools.product([0, 1], repeat=2):
    for batches in itertools.product(range(7), repeat=2):
        if any(batch > 6 * active for batch, active in zip(batches, machine, strict=True)):
            continue
        overtime = max(0.0, 28 - 4 * sum(batches))
        if overtime > 4:
            continue
        cost = 10 * sum(machine) + 3 * sum(batches) + 2 * overtime**2
        candidate = (cost, machine, batches, overtime)
        best = candidate if best is None or candidate < best else best

print("brute-force optimum:", best)
assert factory_result.score["feasible"]
assert abs(factory_result.score["value"] - best[0]) < 1e-5

## 5. Interactive diagnostics and a portable report

The dashboard combines convergence, typed variable domains, constraint health, and annealing dynamics. The exported HTML is self-contained and embeds the result as JSON.

In [ ]:
viz.plot_result_dashboard(factory_result, factory, backend="plotly")

report_path = qqa.save_html_report(
    factory_result, factory, Path("factory-optimization-report.html")
)
print("saved:", report_path)

try:
    from google.colab import files

    files.download(str(report_path))
except ImportError:
    pass